
# Clase paso a paso: Regresión Lineal con datos de ventas de cafetería

En este cuaderno vamos a construir un modelo de **regresión lineal** utilizando datos de transacciones de un café. La regresión lineal es una técnica estadística y de aprendizaje automático que modela la relación entre una variable dependiente (lo que queremos predecir) y una o varias variables independientes (características). A lo largo del cuaderno:

- Explicaremos en detalle **qué hace cada línea de código**.
- Introduciremos conceptos teóricos detrás de cada paso (por ejemplo: por qué normalizar, qué es un *one-hot encoding*, etc.).
- Mostraremos ejemplos sencillos para cimentar las ideas antes de aplicarlas a datos reales.

Al final, tendrás una comprensión completa de cómo limpiar datos, explorar variables, construir un modelo de regresión lineal, evaluarlo y usarlo para predecir sobre nuevos datos.



## Paso 1 — Importar librerías

En Python existen librerías especializadas para análisis de datos y modelado. Cada línea de importación tiene un propósito:

- `import pandas as pd`: importa la biblioteca **pandas** y asigna el alias `pd`. Pandas proporciona estructuras de datos (DataFrame) para manipular tablas con facilidad: lectura, filtrado, agregaciones, etc.
- `import numpy as np`: importa **NumPy** con el alias `np`. NumPy es la biblioteca base para cálculo numérico en Python; nos da acceso a arrays, operaciones matemáticas vectorizadas y funciones estadísticas.
- `import matplotlib.pyplot as plt`: importa el módulo `pyplot` de **Matplotlib**, la biblioteca estándar de visualización. Lo renombramos como `plt` para acceder a funciones de gráficos (p. ej. `plt.plot`, `plt.hist`).
- `import seaborn as sns`: importamos **Seaborn** con alias `sns`. Seaborn se basa en Matplotlib y ofrece funciones de visualización de mayor nivel y apariencia estética. Aunque en este entorno no necesariamente lo usaremos para graficar, lo incluimos para tu ejecución local.

Al definir estos alias (`pd`, `np`, `plt`, `sns`) podemos usar las funciones de cada biblioteca de manera concisa en el código.


In [20]:
!pip install seaborn matplotlib-inline

In [ ]:
# Configuración específica para matplotlib en Jupyter
%matplotlib inline

# Configuración del backend de matplotlib
import matplotlib
matplotlib.use('Agg')  # Backend que no requiere interfaz gráfica
import matplotlib.pyplot as plt

# Reiniciar la configuración de matplotlib después del cambio de backend
plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

print('Matplotlib configurado correctamente para Jupyter.')


In [ ]:
# Prueba rápida para verificar que matplotlib funciona
import numpy as np
x = np.linspace(0, 10, 100)
y = np.sin(x)

plt.figure(figsize=(8, 4))
plt.plot(x, y, 'b-', linewidth=2)
plt.title('Prueba de matplotlib - Función seno')
plt.xlabel('X')
plt.ylabel('sin(X)')
plt.grid(True)
plt.show()

print("✅ Matplotlib está funcionando correctamente!")


In [2]:

# Importamos librerías básicas para análisis y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  # Opcional, se puede usar para graficar localmente

# Configuramos algunos parámetros visuales de matplotlib (tamaño de las figuras y cuadrícula)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

print('Librerías importadas correctamente.')


Librerías importadas correctamente.



## Paso 2 — Cargar el CSV y explorar las primeras filas

El conjunto de datos se encuentra en el archivo `dirty_cafe_sales.csv`. Para trabajar con él:

1. Utilizamos `pd.read_csv(ruta)` para leer el archivo CSV y cargarlo en un **DataFrame** (estructura de tabla de pandas). El parámetro `ruta` debe indicar la ubicación del archivo; en este caso, `/home/oai/share/dirty_cafe_sales.csv`.
2. Guardamos el DataFrame en la variable `df` para manipularlo posteriormente.
3. Usamos `df.head()` para mostrar las primeras filas (por defecto 5). Esto nos da una idea de la estructura y contenido de las columnas sin imprimir todo el conjunto de datos.
4. Las funciones `df.shape` y `df.dtypes` nos permiten ver el tamaño (filas y columnas) y los tipos de datos de cada columna, respectivamente. Muchos valores se leen como `object` (cadenas), aunque deberían ser numéricos. Lo corregiremos más adelante.

Antes de manipular los datos, es importante verificar que se ha cargado correctamente.


In [6]:

# Definimos la ruta del archivo CSV
file_path = 'dirty_cafe_sales.csv'

# Leemos el archivo CSV en un DataFrame
df = pd.read_csv(file_path)

# Visualizamos las primeras 5 filas para comprender la estructura del DataFrame
from IPython.display import display

display(df.head())

# Mostramos el número de filas y columnas y los tipos de datos por columna
print('Dimensiones (filas, columnas):', df.shape)
print('Tipos de datos por columna:')
print(df.dtypes)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


Dimensiones (filas, columnas): (10000, 8)
Tipos de datos por columna:
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object



## Paso 3 — Definir el objetivo del modelo

En la regresión lineal elegimos una **variable objetivo** (también llamada variable dependiente) que queremos predecir a partir de otras características. En este problema, nuestra variable objetivo es **`Total Spent`** (total gastado en la transacción), ya que queremos estimar cuánto se gasta en función del tipo de producto, cantidad comprada, método de pago, ubicación y fecha.

Las **variables independientes** (o características) incluirán:

- `Item`: producto comprado (café, jugo, etc.).
- `Quantity`: número de unidades.
- `Price Per Unit`: precio unitario del producto.
- `Payment Method`: método de pago utilizado.
- `Location`: dónde se realizó la compra (en tienda, para llevar).
- `Transaction Date`: fecha en que se realizó la transacción.

Más adelante convertiremos la fecha en variables numéricas (año, mes, día, día de la semana). Este planteamiento nos guiará en cómo preparar los datos y construir el modelo.



## Paso 4 — Diagnóstico de valores faltantes y tipos de datos

Antes de entrenar un modelo, debemos asegurarnos de que los datos tengan el formato correcto y entender cuántos valores faltantes hay. Esto se hace con dos funciones:

- `df.isna().sum()`: cuenta el número de valores `NaN` (falta de datos) por columna. Esto nos ayuda a decidir si debemos eliminar filas, imputar valores o tratar categorías especiales.
- `df.dtypes`: nos muestra los tipos de datos. En muchos archivos CSV, las columnas numéricas pueden leerse como tipo `object` (cadena). Posteriormente convertiremos estas columnas a valores numéricos.

Comprender los valores faltantes es clave para no cometer errores de análisis. Por ejemplo, si una columna tiene demasiados valores nulos, podríamos descartarla o buscar otra estrategia de imputación.


In [7]:

# Mostramos el conteo de valores faltantes por columna
faltantes = df.isna().sum()
print('Valores faltantes por columna:', faltantes)

# Mostramos el tipo de datos de cada columna
print('Tipos de datos originales:')
print(df.dtypes)


Valores faltantes por columna: Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64
Tipos de datos originales:
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object



## Paso 5 — Conversión de columnas numéricas

Es habitual que en un CSV los números se interpreten como cadenas de texto (`object` en pandas). Para realizar operaciones aritméticas y modelado necesitamos convertirlas a un tipo numérico (`float` o `int`).

Utilizaremos `pd.to_numeric()` con el parámetro `errors='coerce'` para convertir las columnas `Quantity`, `Price Per Unit` y `Total Spent` a números:

- Si el valor se puede convertir, lo hará sin problemas.
- Si hay caracteres no numéricos o cadenas vacías, se asignará `NaN` (valor faltante) en esas posiciones.

Tras la conversión, podemos utilizar `df[['columna']].describe()` para ver un resumen estadístico (media, desviación estándar, mínimos y máximos) de las columnas numéricas. Esto nos da una idea de su escala y posibles valores atípicos.


In [8]:

# Hacemos una copia del DataFrame para no modificar el original
df_clean = df.copy()

# Lista de columnas que deberían ser numéricas
numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']

# Convertimos cada una a numérico; los errores se convertirán en NaN
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Mostramos estadísticas descriptivas de las columnas numéricas convertidas
print(df_clean[numeric_cols].describe())


          Quantity  Price Per Unit  Total Spent
count  9521.000000     9467.000000  9498.000000
mean      3.028463        2.949984     8.924352
std       1.419007        1.278450     6.009919
min       1.000000        1.000000     1.000000
25%       2.000000        2.000000     4.000000
50%       3.000000        3.000000     8.000000
75%       4.000000        4.000000    12.000000
max       5.000000        5.000000    25.000000



## Paso 6 — Limpieza de categorías inválidas

En las columnas categóricas hay valores como `ERROR` o `UNKNOWN` que indican un dato desconocido o erróneo. Estos valores no representan una categoría real, por lo que los reemplazaremos por `NaN` para tratarlos como valores faltantes. Más adelante podremos decidir si eliminamos filas con demasiados `NaN` o imputamos un valor.

Realizamos la operación para las columnas `Item`, `Payment Method` y `Location`:

1. Recorremos la lista de columnas categóricas.
2. Usamos `.replace({'ERROR': np.nan, 'UNKNOWN': np.nan})` para sustituir cada ocurrencia por un `NaN`.
3. Finalmente, contamos los valores faltantes (`.isna().sum()`) para verificar cuántos valores fueron reemplazados.

Esta limpieza es importante para no considerar `ERROR` o `UNKNOWN` como categorías válidas en el modelo.


In [9]:

# Lista de columnas categóricas a limpiar
categorical_cols = ['Item', 'Payment Method', 'Location']

# Sustituimos 'ERROR' y 'UNKNOWN' por NaN en cada columna categórica
for col in categorical_cols:
    df_clean[col] = df_clean[col].replace({'ERROR': np.nan, 'UNKNOWN': np.nan})

# Mostramos cuántos valores faltantes hay después de la limpieza
for col in categorical_cols:
    print(f"{col}: {df_clean[col].isna().sum()} valores NaN")


Item: 969 valores NaN
Payment Method: 3178 valores NaN
Location: 3961 valores NaN



## Paso 7 — Conversión de la fecha y creación de variables temporales

La columna `Transaction Date` contiene fechas en formato de texto. Es útil convertirla a un objeto de tipo **datetime** para poder extraer componentes como:

- `Year`: año de la transacción (ej. 2023).
- `Month`: mes (1–12).
- `Day`: día del mes (1–31).
- `DayOfWeek`: día de la semana (0–6, donde 0 = lunes y 6 = domingo).

Usaremos `pd.to_datetime()` con `errors='coerce'` para convertir la columna a fechas. Luego, utilizando la notación `df_clean['Transaction Date'].dt.year`, `dt.month`, `dt.day` y `dt.dayofweek` obtendremos las nuevas columnas. Estos atributos se derivan de la fecha original y son ejemplos de **ingeniería de características** (*feature engineering*), donde generamos variables numéricas a partir de datos categóricos o de tiempo.

La columna original de fecha se mantiene, pero de momento no la usaremos como tal.


In [10]:

# Convertimos la columna 'Transaction Date' a datetime
df_clean['Transaction Date'] = pd.to_datetime(df_clean['Transaction Date'], errors='coerce')

# Creamos nuevas columnas basadas en la fecha
df_clean['Year'] = df_clean['Transaction Date'].dt.year
df_clean['Month'] = df_clean['Transaction Date'].dt.month
df_clean['Day'] = df_clean['Transaction Date'].dt.day
df_clean['DayOfWeek'] = df_clean['Transaction Date'].dt.dayofweek

# Mostramos las primeras filas de las columnas de fecha para verificar
print(df_clean[['Transaction Date','Year','Month','Day','DayOfWeek']].head())


  Transaction Date    Year  Month   Day  DayOfWeek
0       2023-09-08  2023.0    9.0   8.0        4.0
1       2023-05-16  2023.0    5.0  16.0        1.0
2       2023-07-19  2023.0    7.0  19.0        2.0
3       2023-04-27  2023.0    4.0  27.0        3.0
4       2023-06-11  2023.0    6.0  11.0        6.0



## Paso 8 — Análisis exploratorio: distribuciones de variables numéricas

El **análisis exploratorio de datos (EDA)** nos ayuda a comprender cómo se distribuyen las variables y a detectar anomalías. En este paso:

- Seleccionamos las columnas numéricas (`Quantity`, `Price Per Unit`, `Total Spent`).
- Utilizamos `hist()` de pandas para dibujar un histograma de cada columna. Un histograma divide los datos en "bins" (intervalos) y muestra cuántos valores caen en cada intervalo.
- Ajustamos el tamaño de la figura con `figsize` y el número de bins con `bins=20` para visualizar mejor la distribución.

Observar estos histogramas nos puede sugerir, por ejemplo, si los datos están sesgados (asimétricos) o si hay valores extremos.


In [21]:


import matplotlib.pyplot as plt
import seaborn as sns

# Seleccionamos las columnas numéricas que queremos analizar
num_cols = ['Quantity','Price Per Unit','Total Spent']

# --- Opción 1: usando Pandas + Matplotlib ---
df_clean[num_cols].hist(bins=20, figsize=(12, 4), edgecolor='black')
plt.suptitle('Distribución de variables numéricas (Pandas + Matplotlib)', y=1.05, fontsize=14)
plt.tight_layout()
plt.show()

# --- Opción 2: usando Seaborn para más estilo ---
fig, axes = plt.subplots(1, len(num_cols), figsize=(15,4))

for i, col in enumerate(num_cols):
    sns.histplot(data=df_clean, x=col, bins=20, kde=True, ax=axes[i], color="skyblue")
    axes[i].set_title(f'Distribución de {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')

plt.suptitle('Distribución de variables numéricas (Seaborn)', y=1.05, fontsize=14)
plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'matplotlib_inline.backend_inline'; 'matplotlib_inline' is not a package


## Paso adicional — Mapa de correlación

Para comprender la relación entre las variables numéricas y cómo se correlacionan entre sí, calcularemos una **matriz de correlación** y la representaremos gráficamente. La correlación mide la fuerza y la dirección de la relación lineal entre dos variables (rango de −1 a 1). Valores cercanos a 1 indican fuerte correlación positiva, valores cercanos a −1 indican fuerte correlación negativa y valores cercanos a 0 indican poca o ninguna correlación lineal.

El mapa de calor (heatmap) que crearemos nos permite visualizar esta matriz de correlación de forma intuitiva. Aunque existen bibliotecas como `seaborn` que simplifican este tipo de gráficos, a continuación utilizaremos **matplotlib** para mantener la compatibilidad con entornos más restringidos y respetar la política del entorno.


In [22]:

# Seleccionamos las columnas numéricas del DataFrame limpio
df_num = df_clean[['Quantity', 'Price Per Unit', 'Total Spent', 'Year', 'Month', 'DayOfWeek']]

# Calculamos la matriz de correlación
correlation_matrix = df_num.corr()
print('Matriz de correlación:', correlation_matrix)

# Creamos un mapa de calor usando matplotlib
fig, ax = plt.subplots(figsize=(6,5))
cax = ax.matshow(correlation_matrix, cmap='viridis')
plt.title('Mapa de correlación entre variables numéricas', pad=20)
ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_yticks(range(len(correlation_matrix.columns)))
ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha='left')
ax.set_yticklabels(correlation_matrix.columns)

# Añadimos una barra de color
fig.colorbar(cax)

# Anotamos los valores en cada celda para una mejor interpretación
for (i, j), val in np.ndenumerate(correlation_matrix.values):
    ax.text(j, i, f"{val:.2f}", va='center', ha='center', color='white' if abs(val) > 0.5 else 'black')

plt.show()


Matriz de correlación:                 Quantity  Price Per Unit  Total Spent  Year     Month  \
Quantity        1.000000        0.006078     0.704310   NaN -0.003688   
Price Per Unit  0.006078        1.000000     0.647489   NaN -0.011588   
Total Spent     0.704310        0.647489     1.000000   NaN -0.009235   
Year                 NaN             NaN          NaN   NaN       NaN   
Month          -0.003688       -0.011588    -0.009235   NaN  1.000000   
DayOfWeek       0.000177       -0.003015    -0.002541   NaN  0.021567   

                DayOfWeek  
Quantity         0.000177  
Price Per Unit  -0.003015  
Total Spent     -0.002541  
Year                  NaN  
Month            0.021567  
DayOfWeek        1.000000  


ModuleNotFoundError: No module named 'matplotlib_inline.backend_inline'; 'matplotlib_inline' is not a package


## Paso 9 — Análisis exploratorio: relación entre variables

Para comprender la relación entre las variables independientes y nuestra variable objetivo, realizaremos gráficos de dispersión (**scatter plots**). Un scatter plot muestra puntos individuales, donde el eje `x` es una variable y el eje `y` es otra. En el contexto de regresión lineal:

- Si la relación es aproximadamente lineal, los puntos se alinearán formando una nube en torno a una línea.
- Si no, puede indicar la necesidad de transformaciones o modelos más complejos.

En este paso representaremos:

- `Quantity` vs `Total Spent`: esperamos ver que, a mayor cantidad de productos, mayor gasto total.
- `Price Per Unit` vs `Total Spent`: un precio unitario mayor debería influir en un total mayor, pero el efecto dependerá de la cantidad.

Los parámetros `alpha=0.4` o similares ajustan la transparencia de los puntos, ayudando a ver la densidad de datos en gráficas con muchos puntos.


In [ ]:

# Gráfica de dispersión entre Quantity y Total Spent
plt.scatter(df_clean['Quantity'], df_clean['Total Spent'], alpha=0.4)
plt.xlabel('Quantity (Cantidad)')
plt.ylabel('Total Spent (Gasto total)')
plt.title('Relación entre Quantity y Total Spent')
plt.show()

# Gráfica de dispersión entre Price Per Unit y Total Spent
plt.scatter(df_clean['Price Per Unit'], df_clean['Total Spent'], alpha=0.4)
plt.xlabel('Price Per Unit (Precio unitario)')
plt.ylabel('Total Spent (Gasto total)')
plt.title('Relación entre Price Per Unit y Total Spent')
plt.show()



## Paso 10 — Mini‑ejemplo de regresión lineal con datos sintéticos

Antes de aplicar la regresión lineal a nuestros datos reales, veremos un ejemplo simple para entender cómo funciona. Generaremos datos artificiales con una relación lineal conocida:

- `x`: valores entre 0 y 10 generados uniformemente.
- `y`: obtenidos con una fórmula `y = a * x + b + ruido`, donde `a` es la pendiente (2.5) y `b` el intercepto (5). El `ruido` añade variabilidad aleatoria para simular mediciones reales.

Utilizaremos `sklearn.linear_model.LinearRegression` para ajustar una recta a estos datos. El método `fit(X_train, y_train)` aprende la pendiente e intercepto que minimizan la suma de los errores cuadráticos. Después, veremos las pendientes e interceptos estimados y dibujaremos la línea ajustada sobre los datos.

Este ejemplo ayuda a comprender lo que hace el modelo en una situación controlada.


In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Generamos datos sintéticos
np.random.seed(42)
X = np.random.rand(100, 1) * 10   # 100 puntos entre 0 y 10
true_slope = 2.5
true_intercept = 5
noise = np.random.randn(100, 1)   # ruido gaussiano

y = true_slope * X + true_intercept + noise  # definimos la relación lineal con ruido

# Dividimos los datos en entrenamiento y prueba (80%/20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Creamos el modelo de regresión lineal
model_simple = LinearRegression()

# Ajustamos el modelo con los datos de entrenamiento
model_simple.fit(X_train, y_train)

# Calculamos predicciones para los datos de prueba
y_pred = model_simple.predict(X_test)

# Imprimimos la pendiente e intercepto aprendidos
print('Pendiente estimada:', model_simple.coef_[0][0])
print('Intercepto estimado:', model_simple.intercept_[0])

# Visualizamos los datos y la línea ajustada
plt.scatter(X, y, label='Datos reales', alpha=0.5)

# Para dibujar la línea de regresión generamos 100 puntos a lo largo de X
x_line = np.linspace(0, 10, 100).reshape(-1, 1)
y_line = model_simple.predict(x_line)

plt.plot(x_line, y_line, color='red', label='Recta ajustada')
plt.title('Ejemplo de regresión lineal simple')
plt.xlabel('Variable independiente (X)')
plt.ylabel('Variable dependiente (y)')
plt.legend()
plt.show()



## Paso 11 — Selección de características y filtrado de filas válidas

Con los datos reales, ahora vamos a preparar un conjunto de variables (características) que usaremos para entrenar nuestro modelo. Estos pasos son:

1. **Elegir las columnas** que serán las variables independientes (`Item`, `Quantity`, `Price Per Unit`, `Payment Method`, `Location`, `Year`, `Month`, `DayOfWeek`) y la variable objetivo (`Total Spent`).
2. **Eliminar filas con valores faltantes** en estas columnas clave. Si una fila tiene `NaN` en alguna de estas columnas, no aportará información completa al modelo. Utilizamos `dropna()` para desechar dichas filas.
3. Guardamos el resultado en `data` y verificamos cuántas filas quedan después del filtrado.

Este filtrado asegura que el modelo reciba entradas completas y consistentes para todas las variables seleccionadas.


In [ ]:

# Definimos la lista de columnas que usaremos como características (features)
features = ['Item', 'Quantity', 'Price Per Unit', 'Payment Method', 'Location', 'Year', 'Month', 'DayOfWeek']

# Definimos la variable objetivo
target = 'Total Spent'

# Construimos un nuevo DataFrame con las columnas de interés
data = df_clean[features + [target]].copy()

# Eliminamos filas con valores faltantes en cualquiera de estas columnas
data = data.dropna()

# Mostramos el tamaño del DataFrame resultante
tamano = data.shape
print('Tamaño después del filtrado (filas, columnas):', tamano)

# Mostramos las primeras filas para confirmar el resultado
data.head()



## Paso 12 — División en conjuntos de entrenamiento y prueba

Para evaluar el desempeño de un modelo debemos separar los datos en dos subconjuntos:

- **Conjunto de entrenamiento (train)**: se utiliza para ajustar el modelo.
- **Conjunto de prueba (test)**: se mantiene aparte y solo se utiliza para evaluar el modelo después del entrenamiento.

Hacer esta división es crucial porque simula la capacidad del modelo para generalizar a datos no vistos. Si evaluáramos el modelo en los mismos datos con los que lo entrenamos, estaríamos midiendo qué tan bien memorizó el conjunto de entrenamiento, no qué tan bien predice casos nuevos.

Usaremos `train_test_split()` de scikit‑learn:

- `test_size=0.2` indica que el 20% de los datos se reserva para prueba y el 80% para entrenamiento.
- `random_state=42` fija una semilla para que la partición sea reproducible (esto significa que cada vez que corras el código con la misma semilla obtendrás la misma división).

Separaremos `X` (features) de `y` (variable objetivo) antes de hacer el split.


In [ ]:

from sklearn.model_selection import train_test_split

# Separación de variables independientes (X) y objetivo (y)
X = data[features]
y = data[target]

# División del dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Tamaño de entrenamiento:', X_train.shape)
print('Tamaño de prueba:', X_test.shape)



## Paso 13 — Crear un Pipeline de preprocesamiento y el modelo de regresión

La regresión lineal en scikit‑learn no puede manejar datos categóricos ni variables en diferentes escalas directamente. Necesitamos preprocesar las variables:

1. **Escalar las variables numéricas**: Usamos `StandardScaler()` para que las variables numéricas (`Quantity`, `Price Per Unit`, `Year`, `Month`, `DayOfWeek`) tengan media cero y varianza uno. Esto evita que variables con mayores magnitudes dominen la regresión.
2. **Codificar variables categóricas**: Utilizamos `OneHotEncoder()` para convertir cada categoría de `Item`, `Payment Method` y `Location` en una columna binaria (0/1). El parámetro `handle_unknown='ignore'` hace que valores no vistos durante el ajuste se manejen adecuadamente.

Para aplicar diferentes transformaciones a distintas columnas usamos `ColumnTransformer`, que recibe una lista de (nombre, transformador, columnas) y aplica cada transformador solo a las columnas especificadas. De esta forma, las columnas numéricas se escalan y las categóricas se convierten a variables dummy.

Finalmente, combinamos el preprocesamiento con el modelo de regresión lineal en un `Pipeline`. Un pipeline se ejecuta en serie: primero preprocesa los datos y luego ajusta el modelo. Esto garantiza que cualquier nueva muestra pase por las mismas transformaciones antes de la predicción.


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Columnas numéricas y categóricas
num_features = ['Quantity', 'Price Per Unit', 'Year', 'Month', 'DayOfWeek']
cat_features = ['Item', 'Payment Method', 'Location']

# Definimos el transformador para las columnas numéricas: escalado estándar
numeric_transformer = StandardScaler()

# Definimos el transformador para las columnas categóricas: one-hot encoding
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Combinamos ambos transformadores en un ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),  # aplica StandardScaler a num_features
        ('cat', categorical_transformer, cat_features)  # aplica OneHotEncoder a cat_features
    ]
)

# Elegimos la regresión lineal como modelo final
regressor = LinearRegression()

# Construimos el pipeline: primero preprocesa y luego entrena el modelo
model = Pipeline(steps=[('preprocessor', preprocessor),
                       ('regressor', regressor)])

print('Pipeline de preprocesamiento y modelo configurado.')



## Paso 14 — Entrenar el modelo

Con el pipeline listo, ajustamos el modelo usando los datos de entrenamiento (`X_train`, `y_train`). El método `fit()` aplica las transformaciones del preprocesamiento y luego ajusta la regresión lineal.

Durante el entrenamiento, el algoritmo de regresión lineal calcula los coeficientes (pesos) que multiplican a cada variable y el intercepto (ordenada al origen) para minimizar el error cuadrático medio entre las predicciones y los valores reales del conjunto de entrenamiento.

Cuando finaliza el entrenamiento, el pipeline almacena el preprocesador ajustado (por ejemplo, los promedios y desviaciones usadas para escalar) y el modelo entrenado para su uso posterior.


In [ ]:

# Ajustamos el pipeline con el conjunto de entrenamiento
model.fit(X_train, y_train)

print('Modelo entrenado correctamente.')



## Paso 15 — Evaluar el modelo con métricas

Para medir el desempeño del modelo evaluaremos varias métricas sobre los datos de entrenamiento y prueba:

- **Error Absoluto Medio (MAE)**: promedio de las diferencias absolutas entre las predicciones y los valores reales. Se interpreta en las mismas unidades que la variable objetivo.
- **Error Cuadrático Medio (MSE)**: promedio de los errores al cuadrado. Penaliza más los errores grandes. Su raíz cuadrada (RMSE) devuelve a la misma escala que la variable objetivo.
- **Coeficiente de determinación (R²)**: mide la proporción de la varianza de la variable objetivo explicada por las variables independientes. Toma valores entre 0 y 1 (aunque puede ser negativo si el modelo es muy malo); un valor cercano a 1 indica buen ajuste.

Usamos las funciones de `sklearn.metrics` para calcular estas métricas. Primero obtenemos las predicciones (`predict()`) en ambos conjuntos, y luego calculamos MAE, RMSE y R².

Comparar las métricas de entrenamiento y prueba nos ayuda a detectar **sobreajuste** (cuando el modelo va muy bien en entrenamiento pero mal en prueba).


In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Obtenemos las predicciones en entrenamiento y prueba
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculamos métricas de entrenamiento
mae_train = mean_absolute_error(y_train, y_pred_train)
rmse_train = mean_squared_error(y_train, y_pred_train, squared=False)
r2_train = r2_score(y_train, y_pred_train)

# Calculamos métricas de prueba
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = mean_squared_error(y_test, y_pred_test, squared=False)
r2_test = r2_score(y_test, y_pred_test)

# Mostramos los resultados
print('--- Conjunto de entrenamiento ---')
print(f'MAE : {mae_train:.2f}')
print(f'RMSE: {rmse_train:.2f}')
print(f'R²  : {r2_train:.2f}')

print('
--- Conjunto de prueba ---')
print(f'MAE : {mae_test:.2f}')
print(f'RMSE: {rmse_test:.2f}')
print(f'R²  : {r2_test:.2f}')



## Paso 16 — Visualizar predicciones vs valores reales

Una forma intuitiva de evaluar el modelo es comparar las predicciones con los valores reales en el conjunto de prueba. Dibujamos:

- Puntos (`scatter`) donde el eje `x` son los valores reales (`y_test`) y el eje `y` son las predicciones (`y_pred_test`).
- La línea `y = x` (diagonal), que representa las predicciones perfectas. Si todos los puntos estuvieran sobre esta línea, el modelo acertaría todos los valores.

Cuanto más cerca estén los puntos de la diagonal, mejor es el ajuste. Si la nube de puntos se dispersa mucho o forma un patrón claramente alejado de la diagonal, indica que el modelo tiene errores sistemáticos.


In [ ]:

# Dibujamos las predicciones versus los valores reales
plt.scatter(y_test, y_pred_test, alpha=0.4)
plt.xlabel('Valores reales (y_test)')
plt.ylabel('Predicciones (y_pred_test)')
plt.title('Predicciones vs Valores reales en el conjunto de prueba')

# Dibujamos la línea ideal y = x
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Línea ideal')
plt.legend()
plt.show()



## Paso adicional — Selección de características con `SelectKBest`

La **selección de características** es el proceso de elegir un subconjunto de variables independientes que aportan mayor información a la variable objetivo. Utilizar menos características puede simplificar el modelo, reducir el sobreajuste y disminuir el tiempo de entrenamiento.

En este ejemplo usaremos `SelectKBest` de `sklearn.feature_selection` con el estadístico **`f_regression`** para seleccionar las *k* mejores características numéricas. Este método evalúa la relación lineal entre cada variable independiente y la variable dependiente (`Total Spent`), devolviendo un valor `F` y un *p-value*. Seleccionaremos las características con mayores valores `F` (mayor correlación lineal con el objetivo). Recuerda que este método solo se aplica a variables numéricas; las variables categóricas se transforman mediante `OneHotEncoder` en el pipeline.

Ajustaremos `SelectKBest` después de que el `ColumnTransformer` genere la matriz de características completas (numéricas escaladas + categóricas codificadas). De forma orientativa, elegiremos las 10 variables más relevantes.

**Nota**: La selección de características debe aplicarse solamente en el conjunto de entrenamiento para evitar sesgo; luego se aplica la misma máscara de selección al conjunto de prueba.


In [ ]:

from sklearn.feature_selection import SelectKBest, f_regression

# Aplicamos el preprocesamiento para obtener la matriz de características completa
data_full = model.named_steps['preprocessor'].fit_transform(X_train)

# Aplicamos SelectKBest para seleccionar las mejores k características
k = 10  # número de características a seleccionar
selector = SelectKBest(score_func=f_regression, k=k)
selector.fit(data_full, y_train)

# Obtenemos las máscaras de selección y nombres de columnas transformadas
selected_mask = selector.get_support()

# Extracción de nombres después del preprocesamiento (numéricas + one-hot)
preprocessor = model.named_steps['preprocessor']

# Nombres numéricos
num_names = num_features

# Nombres categóricos codificados
cat_names = list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features))

all_names = num_names + cat_names

# Seleccionamos nombres de las características elegidas
selected_features = [name for flag, name in zip(selected_mask, all_names) if flag]

print(f'Seleccionadas {len(selected_features)} características:
', selected_features)

# (Opcional) Ajustar un nuevo modelo usando solo estas características podría mejorar la interpretación y eficiencia.



## Paso 17 — Interpretar la importancia de las variables (coeficientes)

En una regresión lineal, los **coeficientes** indican cuánto cambia la predicción cuando una variable cambia, manteniendo el resto constante. Valores altos en valor absoluto sugieren que la variable tiene más influencia sobre la predicción.

Debido al preprocesamiento (`StandardScaler` y `OneHotEncoder`), necesitamos recuperar los nombres de las columnas transformadas para asociar cada coeficiente con su variable original:

1. Accedemos al preprocesador (`model.named_steps['preprocessor']`) y extraemos los nombres de las columnas numéricas y de las columnas codificadas a partir de `OneHotEncoder` (`get_feature_names_out`).
2. Concatenamos esos nombres en una lista, en el mismo orden en que el modelo los ha procesado.
3. Creamos un `Series` de pandas con los coeficientes del modelo (`model.named_steps['regressor'].coef_`) y los nombres como índices.
4. Ordenamos por el valor absoluto del coeficiente para ver cuáles tienen mayor impacto.

Ten en cuenta que los coeficientes se interpretan después de escalamiento; por ello, aunque una categoría tenga un coeficiente grande, su variabilidad puede ser baja.


In [ ]:

import pandas as pd
import numpy as np

# Extraemos los nombres de las columnas transformadas
preprocessor = model.named_steps['preprocessor']

# Nombres de variables numéricas (no cambian)
num_feature_names = num_features

# Nombres de variables categóricas codificadas
cat_transformer = preprocessor.named_transformers_['cat']
cat_feature_names = list(cat_transformer.get_feature_names_out(cat_features))

# Concatenamos todos los nombres en el orden en que se aplican
all_feature_names = num_feature_names + cat_feature_names

# Extraemos los coeficientes del modelo lineal
coefs = model.named_steps['regressor'].coef_

# Creamos una serie con nombres y coeficientes
coef_series = pd.Series(coefs, index=all_feature_names)

# Ordenamos por valor absoluto descendente
coef_importance = coef_series.reindex(coef_series.abs().sort_values(ascending=False).index)

# Mostramos los 15 coeficientes más influyentes
coef_importance.head(15)



## Paso 18 — Probar el modelo con una transacción de ejemplo

Una vez entrenado y evaluado el modelo, podemos utilizarlo para predecir el gasto total en una nueva transacción. Para ello:

1. Creamos un `DataFrame` con la misma estructura que `X_train`, es decir, contiene todas las columnas de características (`Item`, `Quantity`, etc.) en el mismo orden.
2. Rellenamos los valores de ejemplo, por ejemplo:

   - `Item`: "Coffee".
   - `Quantity`: 2 unidades.
   - `Price Per Unit`: 3.50.
   - `Payment Method`: "Credit Card".
   - `Location`: "In-store".
   - `Year`, `Month`, `DayOfWeek`: correspondientes a la fecha (al no usar la columna de fecha original, debemos especificar estos valores).

3. Llamamos a `model.predict()` pasando el DataFrame. Esta función aplicará el mismo preprocesamiento y devolverá la predicción para `Total Spent`.

Mostramos la predicción junto con los valores de entrada. Esta es la forma de aplicar un modelo entrenado a nuevos datos.


In [ ]:

# Definimos una transacción de prueba (asegúrate de usar valores coherentes)
ejemplo = pd.DataFrame([
    {
        'Item': 'Coffee',
        'Quantity': 2,
        'Price Per Unit': 3.5,
        'Payment Method': 'Credit Card',
        'Location': 'In-store',
        'Year': 2025,
        'Month': 8,
        'DayOfWeek': 2  # 2 = martes
    }
])

# Predecimos el total gastado para esta transacción
y_pred_ejemplo = model.predict(ejemplo)[0]

# Mostramos la fila de entrada y la predicción
print('Transacción de ejemplo:')
print(ejemplo)
print(f'Predicción de Total Spent: {y_pred_ejemplo:.2f}')



## Paso 19 — Conclusiones y pasos siguientes

En este cuaderno hemos seguido un flujo completo para construir y entender un modelo de regresión lineal:

1. **Importación y carga de datos**: leímos un CSV y exploramos su estructura.
2. **Diagnóstico y limpieza**: convertimos columnas numéricas, reemplazamos valores inválidos y creamos nuevas variables temporales.
3. **Análisis exploratorio**: examinamos la distribución de las variables y las relaciones entre ellas para justificar la elección de un modelo lineal.
4. **Mini‑ejemplo didáctico**: practicamos con un conjunto de datos sintético para entender la regresión lineal sin complicaciones.
5. **Preprocesamiento**: aplicamos escalado de variables numéricas y codificación `one-hot` para variables categóricas mediante un `ColumnTransformer`.
6. **Modelo**: entrenamos una regresión lineal a través de un `Pipeline`, asegurando que todas las transformaciones se realizan correctamente.
7. **Evaluación**: calculamos métricas de error y visualizamos predicciones vs valores reales para evaluar el ajuste.
8. **Interpretación**: analizamos los coeficientes para entender qué variables influyen más en el total gastado.
9. **Predicción sobre nuevos datos**: demostramos cómo usar el modelo para estimar el gasto en una transacción hipotética.

### Próximos pasos sugeridos

- Probar modelos de regresión con regularización (Ridge, Lasso) para manejar mejor la multicolinealidad y reducir el sobreajuste.
- Usar validación cruzada (`cross_val_score`) para obtener una estimación más robusta del desempeño.
- Explorar modelos no lineales (árboles de decisión, Random Forest) si se sospecha que la relación entre variables y `Total Spent` no es estrictamente lineal.
- Añadir más características, como la hora del día, promociones, día festivo, etc., que puedan explicar mejor la variabilidad en el gasto.

Este cuaderno debería servir como guía robusta y detallada para aprender y enseñar regresión lineal, analizando cada paso y concepto a profundidad.
